In [ ]:
import kagglehub
ankurzing_sentiment_analysis_for_financialnews_path = kagglehub.dataset_download('ankurzing/sentiment-analysis-for-financial-news')

print(f'Data source import complete at {ankurzing_sentiment_analysis_for_financialnews_path}')

In [ ]:
import pandas as pd
import os

def load_and_unify_fintech_data():
    # 1. Define the correct path for Kaggle
    # The directory structure usually follows: /kaggle/input/[dataset-name]/[filename]
    file_path = '/kaggle/input/datasets/ankurzing/sentiment-analysis-for-financial-news/all-data.csv'
    
    # Safety check: List files if you are unsure of the exact name
    # print(os.listdir('/kaggle/input/sentiment-analysis-for-financial-news/'))

    # 2. Read the data directly from the input folder
    # Financial PhraseBank uses 'latin-1' encoding
    df_jargon = pd.read_csv(file_path, encoding='latin-1', header=None, names=['sentiment', 'text'])
    
    # 3. Map sentiments to numerical labels
    df_jargon['label'] = df_jargon['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})
    
    # 4. Clean and Shuffle
    # We drop any rows that failed to map (NaN) and shuffle the dataset
    df_jargon = df_jargon[['text', 'label']].dropna()
    combined_df = df_jargon.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return combined_df

# Execute
df = load_and_unify_fintech_data()
print(f"Total Unified Records: {len(df)}")
print(df.head(50))

In [ ]:
import re

def clean_and_normalize(text):
  """1. Data Cleaning and Normalization"""
  text = text.lower()
  # # Map Emojis before removing symbols
  # for emoji, word in self.emoji_map.items():
  #     text = text.replace(emoji, f" {word} ")

  # Remove URLs and extra symbols, but keep % and $ for context
  text = re.sub(r'http\S+|www\S+', '', text)
  text = re.sub(r'[^a-zA-Z0-9\s%$]', '', text)
  return " ".join(text.split())

df['text'] = df['text'].apply(clean_and_normalize)
print(df.head(50))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

# 1. FIX: Assign 'x' to 'hue' and set 'legend=False' to satisfy new Seaborn requirements
ax = sns.countplot(x='label', data=df, hue='label', palette="viridis", legend=False)

# 2. FIX: Set ticks explicitly before setting labels to avoid the FixedLocator warning
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Negative (0)', 'Neutral (1)', 'Positive (2)'])

plt.title("Distribution of Fintech Sentiments", fontsize=14)
plt.ylabel("Number of Records")

# Add count labels on top of each bar
for p in ax.patches:
    if p.get_height() > 0: # Ensure we only label bars that exist
        ax.annotate(f'{int(p.get_height())}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 9), 
                    textcoords = 'offset points')

plt.show()

In [ ]:
import pandas as pd

def get_balanced_subset(df, total_samples=1000):
    # Calculate samples needed per class
    n_per_class = total_samples // 3 
    
    # Create subsets for each label
    neg_df = df[df['label'] == 0]
    neu_df = df[df['label'] == 1]
    pos_df = df[df['label'] == 2]
    
    # Randomly sample from each
    # .sample(n) picks exactly 'n' records
    neg_sample = neg_df.sample(n=n_per_class, random_state=42)
    neu_sample = neu_df.sample(n=n_per_class, random_state=42)
    pos_sample = pos_df.sample(n=n_per_class, random_state=42)
    
    # Combine and shuffle
    balanced_df = pd.concat([neg_sample, neu_sample, pos_sample])
    return balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 1. Create the balanced 1000-row dataframe
df_balanced = get_balanced_subset(df, total_samples=1000)

print("New Class Distribution:")
print(df_balanced['label'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Split into Train, Validation, and Test
# First, split into training (80%) and temporary (20% for test)
train_val_df, test_df = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label'])

# Then, split the training_val_df into actual training (80% of remaining) and validation (20% of remaining)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df['label']) # 0.25 * 0.8 = 0.2, so 20% for validation

# Convert to Hugging Face Dataset format
ds_train = Dataset.from_pandas(train_df[['text', 'label']])
ds_val = Dataset.from_pandas(val_df[['text', 'label']])
ds_test = Dataset.from_pandas(test_df[['text', 'label']])

print(f"Total samples after sampling: {len(df_balanced)}")
print(f"Training samples: {len(ds_train)}")
print(f"Validation samples: {len(ds_val)}")
print(f"Test samples: {len(ds_test)}")


In [ ]:
import torch
from torch import nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Define weights: [Negative: 2.0, Neutral: 1.0, Positive: 1.0]
        # This forces the model to prioritize getting Negatives (and Sarcasm) right.
        weights = torch.tensor([2.0, 1.0, 1.0]).to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 1. Check for GPU (CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Move your model to the GPU
model.to(device)

# Add your jargon as dedicated tokens
special_tokens_dict = {'additional_special_tokens': ['kyc', 'apr', 'overdraft', 'low_apr', 'under_apr', 'fast_kyc']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))

# After resizing, you can optionally initialize new weights to the mean of existing ones
with torch.no_grad():
    embeddings = model.get_input_embeddings()
    # Average of all existing embeddings
    mean_embedding = embeddings.weight[:-num_added_toks].mean(dim=0)
    # Set new tokens to this mean
    for i in range(num_added_toks):
        embeddings.weight[-1-i] = mean_embedding

def preprocess(examples):
    return tokenizer(examples["text"], return_tensors="pt", truncation=True, padding=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_tokenized = ds_train.map(preprocess, batched=True)
val_tokenized = ds_val.map(preprocess, batched=True)

# Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Training Arguments
args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    fp16=True,                # Use Mixed Precision (faster on GPU)
    use_cpu=False,            # Ensure CPU is NOT used
    per_device_train_batch_size=10,
    num_train_epochs=15,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
# trainer = WeightedTrainer(
#     model=model,
#     args=args, # Use your previous TrainingArguments
#     train_dataset=train_tokenized,
#     eval_dataset=val_tokenized,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics
# )

trainer.train()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Get predictions on your test set
predictions = trainer.predict(val_tokenized)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# 2. Compute the matrix
cm = confusion_matrix(y_true, y_pred)
labels = ['Negative', 'Neutral', 'Positive']

# 3. Plot using Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted Sentiment')
plt.ylabel('Actual Sentiment')
plt.title('Fintech Sentiment Confusion Matrix')
plt.show()

In [ ]:
import torch

def predict(text):
    # 1. Ensure the model is on the correct device (GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model.to(device)
    model.eval() # Set to evaluation mode

    # 2. Preprocess and Tokenize
    # Ensure this matches the tokenizer you used during training
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True
    ).to(device)

    # 3. Inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    # 4. Map ID to Label String
    # 0: negative, 1: neutral, 2: positive
    prediction_id = torch.argmax(logits, dim=-1).item()
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    
    return label_map[prediction_id]

In [ ]:
import torch
# Take 10 samples from validation
for i in range(10):
    text = ds_test[i]['text']
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits
    print(f"Logits: {logits.cpu().numpy()} | Pred: {torch.argmax(logits).item()}")

In [ ]:
# Assuming your columns are named 'text' and 'label'
print(f"{'Text':<70} | {'Expected':<10} | {'Predicted':<10} | {'Status'}")
print("-" * 110)

# If ds_test is a list of dictionaries (standard for HF Datasets)
for row in ds_test:
    text = row['text']
    # Map the numerical label back to a string for comparison
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    expected = label_map[row['label']]
    
    lower_text = clean_and_normalize(text)
    predicted = predict(lower_text).lower()
    
    status = "✅" if predicted == expected else "❌"
    print(f"{text[:68]:<70} | {expected:<10} | {predicted:<10} | {status}")